# FantasAI LLM-Powered Chat API

## Architecture

This notebook builds a complete RAG (Retrieval-Augmented Generation) chat system for the FantasAI Fantasy Football app that combines:

* **🤖 LLM Intelligence**: Databricks Foundation Models (DBRX/Llama) for natural language understanding
* **🎯 ML Predictions**: 4 trained LightGBM models (QB/RB/WR/TE) for weekly player projections
* **🔍 Vector Search**: Semantic search over 28,826 player records for context retrieval
* **📊 Data Layer**: Gold layer historical stats and analytics tables

## Capabilities

* Natural language Q&A about players and matchups
* Weekly fantasy point predictions
* Player comparisons and start/sit recommendations
* Historical performance analysis
* Injury status and trend insights

## Deployment

The final chat function will be packaged as an MLflow model and deployed to a Model Serving endpoint, exposing a REST API for your front-end Fantasy Football app.

---

In [0]:
%pip install --quiet databricks-vectorsearch databricks-sdk openai mlflow lightgbm
dbutils.library.restartPython()

In [0]:
import mlflow
import json
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from openai import OpenAI
import pandas as pd
import numpy as np

# Unity Catalog Configuration
CATALOG = "main"
SCHEMA = "fantasai"

# SQL Warehouse Configuration (for Model Serving)
SQL_WAREHOUSE_ID = "d0d577334268e08f"  # Serverless Starter Warehouse

# Vector Search Configuration
VECTOR_SEARCH_ENDPOINT = "fantasai-vs-endpoint"  # Corrected: using hyphen
VECTOR_INDEX_NAME = f"{CATALOG}.{SCHEMA}.fantasai_player_index"

# Model Run IDs from Phase 3 (ml_model_registration notebook)
MODEL_RUN_IDS = {
    'QB': '912be789d4524a0a9f926431fc8a88fd',
    'RB': 'ae3a7c9fb600494aa93efd7cc22ef7cb',
    'WR': '237753d24cc24d29a85a7cdf526a5780',
    'TE': 'c18fcbf8c7b14aa7a8a1a47760048fe0'
}

# Feature columns (42 features from ml_player_features)
FEATURE_COLS = [
    'rolling_3g_avg', 'rolling_5g_avg', 'season_avg_to_date', 'momentum_score',
    'wow_change', 'scoring_streak', 'rolling_10g_avg', 'rolling_5g_stddev',
    'form_variance_ratio', 'is_early_season', 'is_mid_season', 'is_late_season',
    'is_playoff_weeks', 'weeks_into_season', 'games_played_streak', 'weeks_since_last_game',
    'coming_off_bye', 'qb_passing_yards', 'qb_passing_tds', 'qb_attempts',
    'qb_completions', 'rb_carries', 'rb_rushing_yards', 'rb_rushing_tds',
    'rec_targets', 'rec_receptions', 'rec_yards', 'rec_tds',
    'rolling_3g_targets', 'rolling_3g_carries', 'rolling_3g_attempts',
    'career_avg_vs_opponent', 'games_vs_opponent', 'max_points_vs_opponent',
    'recent_avg_vs_opponent', 'team_offensive_strength', 'team_offense_rank',
    'position_share_pct', 'def_points_allowed_avg', 'def_rank_vs_position',
    'season_position_rank', 'season_percentile'
]

# Databricks Foundation Model Configuration
FOUNDATION_MODEL = "databricks-meta-llama-3-3-70b-instruct"  # Corrected: 3.3 not 3.1

print("✓ Configuration loaded")
print(f"  Catalog: {CATALOG}")
print(f"  Schema: {SCHEMA}")
print(f"  SQL Warehouse ID: {SQL_WAREHOUSE_ID}")
print(f"  Vector Index: {VECTOR_INDEX_NAME}")
print(f"  Foundation Model: {FOUNDATION_MODEL}")
print(f"  Prediction Models: {len(MODEL_RUN_IDS)} positions loaded")

In [0]:
print("Loading Vector Search client and ML models...")
print("=" * 70)

# Initialize Vector Search Client
vsc = VectorSearchClient()
print("\n✓ Vector Search client initialized")

# Get Vector Search index
try:
    vs_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_INDEX_NAME
    )
    print(f"✓ Vector Search index connected: {VECTOR_INDEX_NAME}")
except Exception as e:
    print(f"⚠ Warning: Could not connect to Vector Search index: {e}")
    print("  You may need to verify the endpoint and index names.")
    vs_index = None

# Load LightGBM prediction models
prediction_models = {}
for position, run_id in MODEL_RUN_IDS.items():
    model_uri = f"runs:/{run_id}/model"
    try:
        model = mlflow.lightgbm.load_model(model_uri)
        prediction_models[position] = model
        print(f"✓ Loaded {position} model: {run_id[:8]}...")
    except Exception as e:
        print(f"✗ Failed to load {position} model: {e}")

print(f"\n{'='*70}")
print(f"✓ Setup complete: {len(prediction_models)}/{len(MODEL_RUN_IDS)} models loaded")
print(f"{'='*70}")

In [0]:
def get_player_prediction(player_name, position, features_dict=None):
    """
    Get weekly fantasy point prediction for a player.
    
    Args:
        player_name: Player's full name
        position: QB, RB, WR, or TE
        features_dict: Dictionary of 42 feature values (optional)
        
    Returns:
        Dictionary with prediction and confidence info
    """
    if position not in prediction_models:
        return {
            'error': f'No model available for position {position}',
            'player': player_name,
            'position': position
        }
    
    model = prediction_models[position]
    
    # If features not provided, fetch from ml_player_features table
    if features_dict is None:
        try:
            query = f"""
            SELECT {', '.join(FEATURE_COLS)}
            FROM {CATALOG}.{SCHEMA}.ml_player_features
            WHERE player_name = '{player_name}'
            AND position = '{position}'
            ORDER BY season DESC, week DESC
            LIMIT 1
            """
            features_df = spark.sql(query).toPandas()
            
            if len(features_df) == 0:
                return {
                    'error': f'No feature data found for {player_name} ({position})',
                    'player': player_name,
                    'position': position
                }
            
            features_dict = features_df.iloc[0].to_dict()
        except Exception as e:
            return {
                'error': f'Failed to fetch features: {str(e)}',
                'player': player_name,
                'position': position
            }
    
    # Create feature array in correct order
    try:
        feature_values = [features_dict.get(col, 0.0) for col in FEATURE_COLS]
        feature_array = np.array([feature_values])
        
        # Make prediction
        prediction = model.predict(feature_array)[0]
        
        return {
            'player': player_name,
            'position': position,
            'predicted_points': round(float(prediction), 2),
            'confidence': 'high',  # Can be enhanced with prediction intervals
            'features_used': len(FEATURE_COLS)
        }
    except Exception as e:
        return {
            'error': f'Prediction failed: {str(e)}',
            'player': player_name,
            'position': position
        }

# Test the prediction function
test_prediction = get_player_prediction("Patrick Mahomes", "QB")
print("Test Prediction:")
print(json.dumps(test_prediction, indent=2))

In [0]:
def retrieve_player_context(query_text, num_results=5, position_filter=None):
    """
    Retrieve relevant player information from gold layer tables.
    
    Note: Vector Search index uses self-managed embeddings, so we query
    the gold layer tables directly instead.
    
    Args:
        query_text: Natural language query (used to extract player names/keywords)
        num_results: Number of results to return
        position_filter: Optional position filter (QB, RB, WR, TE)
        
    Returns:
        List of relevant player records with context
    """
    try:
        # Extract keywords from query (simple approach - can be enhanced)
        keywords = query_text.lower().replace('?', '').replace(',', ' ').split()
        
        # Build SQL query for recent top performers
        where_clause = ""
        if position_filter:
            where_clause = f"WHERE position = '{position_filter}'"
        
        query = f"""
        SELECT 
            player_name,
            position,
            team,
            season,
            week,
            fantasy_points,
            CONCAT(
                player_name, ' (', position, ') averaged ',
                ROUND(fantasy_points, 1), ' fantasy points in ',
                season, ' week ', week
            ) as player_summary
        FROM {CATALOG}.{SCHEMA}.gold_weekly_stats
        {where_clause}
        ORDER BY season DESC, week DESC, fantasy_points DESC
        LIMIT {num_results}
        """
        
        results_df = spark.sql(query).toPandas()
        
        if len(results_df) == 0:
            return [{'info': 'No player data found'}]
        
        # Convert to list of dictionaries
        context_items = results_df.to_dict('records')
        
        return context_items
        
    except Exception as e:
        return [{'error': f'Data retrieval failed: {str(e)}'}]

# Test retrieval function
test_context = retrieve_player_context("top performing quarterback", num_results=3, position_filter="QB")
print("Test Retrieval:")
for i, item in enumerate(test_context[:2], 1):
    if 'error' not in item:
        print(f"\n{i}. {item.get('player_name')} ({item.get('position')})")
        print(f"   Team: {item.get('team')}")
        print(f"   {item.get('season')} Week {item.get('week')}: {item.get('fantasy_points'):.1f} pts")

In [0]:
def fantasai_chat(user_question, conversation_history=None):
    """
    Main chat function that combines LLM, Vector Search, and ML predictions.
    
    Args:
        user_question: User's natural language question
        conversation_history: Optional list of prior messages
        
    Returns:
        Dictionary with answer and metadata
    """
    # Initialize OpenAI client for Databricks Foundation Models
    client = OpenAI(
        api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
        base_url=f"{dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()}/serving-endpoints"
    )
    
    # Step 1: Retrieve relevant context from Vector Search
    context_results = retrieve_player_context(user_question, num_results=5)
    context_text = "\n\n".join([
        f"Player: {item.get('player_name', 'N/A')} ({item.get('position', 'N/A')})\n"
        f"Team: {item.get('team', 'N/A')} | Recent Performance: {item.get('fantasy_points', 'N/A')} pts\n"
        f"Summary: {item.get('summary', 'N/A')}"
        for item in context_results[:3] if 'error' not in item
    ])
    
    # Step 2: Build system prompt with fantasy football expertise
    system_prompt = f"""You are FantasAI, an expert fantasy football assistant with access to:
1. Real-time player predictions from advanced ML models
2. Historical performance data and trends
3. Player statistics and matchup analysis

When answering questions:
- Be conversational and confident but not overconfident
- Provide specific predictions when asked
- Reference recent player performance and trends
- Give actionable start/sit recommendations
- Mention injury concerns or roster changes when relevant

Relevant Player Context:
{context_text}

Always provide data-driven insights backed by the context and predictions available."""
    
    # Step 3: Build conversation messages
    messages = [{"role": "system", "content": system_prompt}]
    
    if conversation_history:
        messages.extend(conversation_history)
    
    messages.append({"role": "user", "content": user_question})
    
    # Step 4: Call foundation model
    try:
        response = client.chat.completions.create(
            model=FOUNDATION_MODEL,
            messages=messages,
            max_tokens=500,
            temperature=0.7
        )
        
        answer = response.choices[0].message.content
        
        # Step 5: Enhance answer with predictions if player names mentioned
        # (This is a simple implementation - can be enhanced with NER)
        predictions_added = []
        for position in ['QB', 'RB', 'WR', 'TE']:
            for item in context_results[:2]:
                if item.get('position') == position and 'error' not in item:
                    player_name = item.get('player_name')
                    if player_name:
                        pred = get_player_prediction(player_name, position)
                        if 'predicted_points' in pred:
                            predictions_added.append(
                                f"\n\n📊 Prediction: {player_name} - {pred['predicted_points']} pts projected"
                            )
                            break
        
        if predictions_added:
            answer += ''.join(predictions_added[:2])  # Limit to 2 predictions
        
        return {
            'answer': answer,
            'context_used': len(context_results),
            'predictions_added': len(predictions_added),
            'model': FOUNDATION_MODEL
        }
        
    except Exception as e:
        return {
            'error': f'Chat failed: {str(e)}',
            'fallback': 'I apologize, but I encountered an error processing your question. Please try again.'
        }

print("✓ Chat function ready")
print("\nExample usage:")
print('  result = fantasai_chat("Should I start Patrick Mahomes or Josh Allen this week?")')

In [0]:
print("Testing FantasAI Chat System")
print("=" * 70)

# Test 1: Player comparison
print("\n🏈 Test 1: Player Comparison\n")
question1 = "Who should I start this week: Josh Allen or Patrick Mahomes?"
result1 = fantasai_chat(question1)
print(f"Q: {question1}")
print(f"\nA: {result1.get('answer', result1.get('fallback', 'No answer'))}")
print(f"\nMetadata: {result1.get('context_used', 0)} context items, {result1.get('predictions_added', 0)} predictions")

# Test 2: Player performance analysis
print("\n" + "=" * 70)
print("\n🏈 Test 2: Performance Analysis\n")
question2 = "Tell me about Travis Kelce's recent performance and outlook"
result2 = fantasai_chat(question2)
print(f"Q: {question2}")
print(f"\nA: {result2.get('answer', result2.get('fallback', 'No answer'))}")

# Test 3: Position-specific advice
print("\n" + "=" * 70)
print("\n🏈 Test 3: Position Advice\n")
question3 = "Which running back should I pick up from waivers?"
result3 = fantasai_chat(question3)
print(f"Q: {question3}")
print(f"\nA: {result3.get('answer', result3.get('fallback', 'No answer'))}")

print("\n" + "=" * 70)
print("\n✓ Chat testing complete")

In [0]:
import cloudpickle
from mlflow.models.signature import infer_signature
from mlflow.pyfunc import PythonModel
import pandas as pd
import numpy as np

class FantasAIChatModel(PythonModel):
    """
    MLflow PyFunc model wrapper for FantasAI chat API.
    Packages all components for deployment to Model Serving.
    
    Uses pre-loaded data as serialized dictionaries (efficient storage).
    """
    
    def __init__(self, prediction_models=None, catalog=None, schema=None, 
                 foundation_model=None, feature_cols=None,
                 recent_stats_dict=None, player_features_dict=None):
        """
        Initialize with configuration and pre-loaded data as dictionaries.
        """
        self.prediction_models = prediction_models or {}
        self.catalog = catalog
        self.schema = schema
        self.foundation_model = foundation_model
        self.feature_cols = feature_cols or []
        
        # Pre-loaded data as dictionaries (more efficient serialization)
        self.recent_stats_dict = recent_stats_dict or {}
        self.player_features_dict = player_features_dict or {}
        
    def predict(self, context, model_input):
        """
        Main prediction method with system authentication and pre-loaded data.
        """
        import traceback
        from databricks.sdk import WorkspaceClient
        from openai import OpenAI
        
        # Initialize SDK clients with system authentication
        try:
            # Standard WorkspaceClient (uses service principal credentials)
            w = WorkspaceClient()
            
            # Get authentication token for OpenAI client
            auth_header = w.config.authenticate().get('Authorization', '')
            api_token = auth_header.replace('Bearer ', '') if auth_header else ''
            
            openai_client = OpenAI(
                api_key=api_token,
                base_url=f"{w.config.host}/serving-endpoints"
            )
            
        except Exception as e:
            return pd.DataFrame({
                'answer': [f"I apologize, but I encountered an authentication error: {str(e)}"],
                'status': ['error']
            })
        
        # Extract question from input
        if isinstance(model_input, pd.DataFrame):
            questions = model_input['question'].tolist()
        else:
            questions = [model_input.get('question', [''])[0]]
        
        answers = []
        statuses = []
        
        for question in questions:
            try:
                # Step 1: Retrieve player context from pre-loaded data
                context_items = []
                try:
                    # Get top 5 recent performers from dictionary
                    recent_stats_list = self.recent_stats_dict.get('data', [])
                    for stat in recent_stats_list[:5]:
                        context_items.append({
                            'player': stat['player_name'],
                            'position': stat['position'],
                            'team': stat['team'],
                            'points': float(stat['points']),
                            'week': int(stat['week'])
                        })
                except Exception as e:
                    context_items = []  # Continue without context if retrieval fails
                
                # Step 2: Get ML predictions for mentioned players
                predictions = []
                player_keywords = ['mahomes', 'allen', 'kelce', 'hill', 'jefferson']
                for keyword in player_keywords:
                    if keyword.lower() in question.lower():
                        try:
                            # Search for player in pre-loaded features dictionary
                            features_list = self.player_features_dict.get('data', [])
                            
                            # Find matching player
                            player_features = None
                            for features in features_list:
                                if keyword.lower() in features['player_name'].lower():
                                    player_features = features
                                    break
                            
                            if player_features:
                                player_name = player_features['player_name']
                                position = player_features['position']
                                
                                # Get features for prediction
                                feature_values = np.array([player_features[col] for col in self.feature_cols]).reshape(1, -1)
                                
                                if position in self.prediction_models:
                                    pred = self.prediction_models[position].predict(feature_values)[0]
                                    predictions.append({
                                        'player': player_name,
                                        'position': position,
                                        'predicted_points': round(float(pred), 2)
                                    })
                        except Exception as e:
                            continue  # Skip if prediction fails
                
                # Step 3: Generate LLM response
                system_prompt = """You are FantasAI, an expert fantasy football advisor. 
                Use the provided context and predictions to give detailed, actionable advice.
                Reference specific statistics and projections when available."""
                
                context_text = "\n".join([
                    f"- {item['player']} ({item['position']}, {item['team']}): "
                    f"{item['points']:.1f} pts in Week {item['week']}"
                    for item in context_items
                ])
                
                predictions_text = "\n".join([
                    f"- {pred['player']} ({pred['position']}): "
                    f"Projected {pred['predicted_points']} pts"
                    for pred in predictions
                ])
                
                user_prompt = f"""Question: {question}

Recent Performance:
{context_text if context_text else 'No recent stats available'}

ML Projections:
{predictions_text if predictions_text else 'No predictions available'}

Please provide detailed fantasy advice based on this information."""
                
                try:
                    response = openai_client.chat.completions.create(
                        model=self.foundation_model,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt}
                        ],
                        max_tokens=500,
                        temperature=0.7
                    )
                    
                    answer = response.choices[0].message.content
                    status = 'success'
                    
                except Exception as llm_error:
                    answer = f"I apologize, but I encountered an error generating a response: {str(llm_error)}"
                    status = 'error'
                
                answers.append(answer)
                statuses.append(status)
                
            except Exception as e:
                answers.append(f"I apologize, but I encountered an error: {str(e)}\n{traceback.format_exc()}")
                statuses.append('error')
        
        return pd.DataFrame({
            'answer': answers,
            'status': statuses
        })


# Test the model class definition
print("Model Serving Preparation")
print("=" * 70)
print()
print("✓ FantasAIChatModel class defined (with dictionary-based data)")
print("✓ Model signature created: inputs: ")
print("  ['question': string (required)]")
print("outputs: ")
print("  ['answer': string (required), 'status': string (required)]")
print("params: ")
print("  None")
print()
print()
print("Next Steps:")
print("  1. Cell 12 will load data and log model with artifacts")
print("  2. Cell 13 will create/update serving endpoint")
print("  3. Cell 14 will test the endpoint")

# Create sample signature and input for model logging
sample_input = pd.DataFrame({
    'question': ['Who should I start this week: Josh Allen or Patrick Mahomes?']
})

sample_output = pd.DataFrame({
    'answer': ['Based on recent performance...'],
    'status': ['success']
})

signature = infer_signature(sample_input, sample_output)

# Deployment Instructions

## Step 1: Log Model to MLflow (Optional - for UC registration)

```python
with mlflow.start_run(run_name="fantasai_chat_api"):
    model_info = mlflow.pyfunc.log_model(
        artifact_path="fantasai_chat_model",
        python_model=FantasAIChatModel(),
        signature=signature,
        input_example=sample_input,
        pip_requirements=[
            "databricks-vectorsearch",
            "databricks-sdk",
            "openai",
            "mlflow",
            "lightgbm",
            "cloudpickle"
        ]
    )
    print(f"Model logged: {model_info.model_uri}")
```

## Step 2: Create Model Serving Endpoint

### Via Databricks UI:
1. Navigate to **Serving** in the left sidebar
2. Click **Create Serving Endpoint**
3. Configure:
   - **Name**: `fantasai-chat-api`
   - **Served Entities**: Select your logged model
   - **Compute Scale**: Start with Small (1-2 endpoints)
   - **Environment**: Enable GPU if using embedding models
4. Click **Create**
5. Wait for endpoint to reach "Ready" state (~5-10 minutes)

### Via Python SDK:

```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

w = WorkspaceClient()

# Create endpoint
endpoint = w.serving_endpoints.create(
    name="fantasai-chat-api",
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=f"{CATALOG}.{SCHEMA}.fantasai_chat_model",  # If registered to UC
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    )
)

print(f"Endpoint created: {endpoint.name}")
```

## Step 3: Get API Token

1. Go to **User Settings** (click your profile icon → Settings)
2. Select **Developer** → **Access tokens**
3. Click **Generate new token**
4. Set expiration and permissions
5. Copy and securely store the token

## Step 4: Test Endpoint

```python
import requests
import os

WORKSPACE_URL = "<your-workspace-url>"  # e.g., https://your-workspace.cloud.databricks.com
API_TOKEN = "<your-token>"

url = f"{WORKSPACE_URL}/serving-endpoints/fantasai-chat-api/invocations"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

data = {
    "inputs": {"question": ["Who should I start at QB this week?"]}
}

response = requests.post(url, headers=headers, json=data)
print(response.json())
```

---

## Alternative: Direct Deployment (Without UC Registration)

If you want to deploy quickly without Unity Catalog:

```python
# Option 1: Deploy from local function
import mlflow.pyfunc

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FantasAIChatModel(),
        signature=signature,
        input_example=sample_input
    )
    
    # Note the run_id for deployment
    run_id = mlflow.active_run().info.run_id
    print(f"Deploy using run_id: {run_id}")
```

Then deploy using the run URI in the serving endpoint configuration.

---

In [0]:
print("FantasAI Chat API - Front-End Integration Examples")
print("=" * 70)

WORKSPACE_URL = "https://your-workspace.cloud.databricks.com"  # Replace with your workspace
ENDPOINT_NAME = "fantasai-chat-api"

print("\n" + "=" * 70)
print("1. Python Client Example")
print("=" * 70)

python_example = '''
import requests
import json

class FantasAIClient:
    def __init__(self, workspace_url, api_token):
        self.workspace_url = workspace_url
        self.api_token = api_token
        self.endpoint_url = f"{workspace_url}/serving-endpoints/fantasai-chat-api/invocations"
    
    def ask(self, question):
        """Send a question to FantasAI chat API."""
        headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json"
        }
        
        data = {
            "inputs": {"question": [question]}
        }
        
        response = requests.post(self.endpoint_url, headers=headers, json=data)
        
        if response.status_code == 200:
            result = response.json()
            return result["predictions"][0]["answer"]
        else:
            raise Exception(f"API Error: {response.status_code} - {response.text}")

# Usage
client = FantasAIClient(
    workspace_url="https://your-workspace.cloud.databricks.com",
    api_token="dapi1234..."
)

answer = client.ask("Should I start Josh Allen or Patrick Mahomes?")
print(answer)
'''

print(python_example)

print("\n" + "=" * 70)
print("2. JavaScript/React Example")
print("=" * 70)

js_example = '''
// FantasAI API Client for React
class FantasAIClient {
  constructor(workspaceUrl, apiToken) {
    this.workspaceUrl = workspaceUrl;
    this.apiToken = apiToken;
    this.endpointUrl = `${workspaceUrl}/serving-endpoints/fantasai-chat-api/invocations`;
  }

  async ask(question) {
    const response = await fetch(this.endpointUrl, {
      method: 'POST',
      headers: {
        'Authorization': `Bearer ${this.apiToken}`,
        'Content-Type': 'application/json'
      },
      body: JSON.stringify({
        inputs: { question: [question] }
      })
    });

    if (!response.ok) {
      throw new Error(`API Error: ${response.status}`);
    }

    const data = await response.json();
    return data.predictions[0].answer;
  }
}

// React Component Example
import React, { useState } from 'react';

function FantasyChat() {
  const [question, setQuestion] = useState('');
  const [answer, setAnswer] = useState('');
  const [loading, setLoading] = useState(false);

  const client = new FantasAIClient(
    process.env.REACT_APP_DATABRICKS_URL,
    process.env.REACT_APP_API_TOKEN
  );

  const handleAsk = async () => {
    setLoading(true);
    try {
      const response = await client.ask(question);
      setAnswer(response);
    } catch (error) {
      console.error('Error:', error);
      setAnswer('Sorry, something went wrong.');
    } finally {
      setLoading(false);
    }
  };

  return (
    <div className="fantasy-chat">
      <input
        type="text"
        value={question}
        onChange={(e) => setQuestion(e.target.value)}
        placeholder="Ask FantasAI..."
      />
      <button onClick={handleAsk} disabled={loading}>
        {loading ? 'Thinking...' : 'Ask'}
      </button>
      {answer && <div className="answer">{answer}</div>}
    </div>
  );
}
'''

print(js_example)

print("\n" + "=" * 70)
print("3. cURL Example")
print("=" * 70)

curl_example = f'''curl -X POST \\
  {WORKSPACE_URL}/serving-endpoints/{ENDPOINT_NAME}/invocations \\
  -H "Authorization: Bearer ${{DATABRICKS_TOKEN}}" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "inputs": {{
      "question": ["Who should I start at running back this week?"]
    }}
  }}'
'''

print(curl_example)

print("\n" + "=" * 70)
print("4. Environment Variables (.env file)")
print("=" * 70)

env_example = '''# .env file for front-end app
REACT_APP_DATABRICKS_URL=https://your-workspace.cloud.databricks.com
REACT_APP_API_TOKEN=dapi1234567890abcdef
REACT_APP_ENDPOINT_NAME=fantasai-chat-api
'''

print(env_example)

print("\n" + "=" * 70)
print("Security Best Practices")
print("=" * 70)
print("""
⚠️  IMPORTANT SECURITY NOTES:

1. **Never expose API tokens in front-end code**
   - Use a backend proxy/middleware to handle API calls
   - Store tokens in environment variables on your server
   - Never commit tokens to version control

2. **Backend Proxy Pattern** (Recommended):
   - Front-end calls your backend API (e.g., /api/fantasai/ask)
   - Backend authenticates the user
   - Backend calls Databricks API with token from env
   - Backend returns response to front-end

3. **Rate Limiting**:
   - Implement rate limiting on your backend
   - Monitor endpoint usage in Databricks console

4. **CORS Configuration**:
   - Configure allowed origins for your serving endpoint
   - Use Databricks workspace settings to set CORS policies
""")

print("\n" + "=" * 70)
print("✓ API integration examples ready for your front-end team!")
print("=" * 70)

In [0]:
import mlflow
import cloudpickle
from mlflow.models.resources import DatabricksServingEndpoint

print("Step 1: Logging FantasAI Chat Model to MLflow with Pre-Loaded Data")
print("=" * 70)

# Set experiment
mlflow.set_experiment("/Users/kingoffrisco@yahoo.com/fantasai_chat_api_deployment")

print("\nLoading LightGBM models as artifacts...")

# Load all 4 LightGBM models BEFORE creating PyFunc wrapper
loaded_models = {}
for position, run_id in MODEL_RUN_IDS.items():
    model_uri = f"runs:/{run_id}/model"
    try:
        model = mlflow.lightgbm.load_model(model_uri)
        loaded_models[position] = model
        print(f"  ✓ Loaded {position} model: {run_id[:8]}...")
    except Exception as e:
        print(f"  ✗ Failed to load {position} model: {e}")
        raise

print(f"\n✓ All {len(loaded_models)} models loaded successfully")

# Load player data as dictionaries (more efficient)
print("\nLoading player data as serialized dictionaries...")

# Load recent stats (top performers from 2025 season) - limit to 100
recent_stats_query = f"""
SELECT player_name, position, team, 
       fantasy_points as points,
       season, week
FROM {CATALOG}.{SCHEMA}.gold_weekly_stats
WHERE season = 2025
ORDER BY week DESC, fantasy_points DESC
LIMIT 100
"""

recent_stats_df = spark.sql(recent_stats_query).toPandas()
recent_stats_dict = {
    'data': recent_stats_df.to_dict('records')
}
print(f"  ✓ Loaded {len(recent_stats_dict['data'])} recent player performance records")

# Load player features (for ML predictions) - limit to recent weeks for key players
# Focus on QBs, RBs, WRs, TEs from the last 4 weeks
player_features_query = f"""
SELECT *
FROM {CATALOG}.{SCHEMA}.ml_player_features
WHERE season = 2025 
  AND week >= 15
  AND position IN ('QB', 'RB', 'WR', 'TE')
ORDER BY season DESC, week DESC, season_avg_to_date DESC
LIMIT 500
"""

player_features_df = spark.sql(player_features_query).toPandas()
player_features_dict = {
    'data': player_features_df.to_dict('records')
}
print(f"  ✓ Loaded {len(player_features_dict['data'])} player feature records")
print(f"  ✓ Data converted to efficient dictionary format")

print("\nCreating PyFunc wrapper with pre-loaded models and data...")

# Create FantasAIChatModel instance with pre-loaded models and data as dicts
fantasai_model = FantasAIChatModel(
    prediction_models=loaded_models,
    catalog=CATALOG,
    schema=SCHEMA,
    foundation_model=FOUNDATION_MODEL,
    feature_cols=FEATURE_COLS,
    recent_stats_dict=recent_stats_dict,
    player_features_dict=player_features_dict
)

print("\n✓ PyFunc model created with dictionary-based data")
print(f"  Recent stats: {len(recent_stats_dict['data'])} records")
print(f"  Player features: {len(player_features_dict['data'])} records")
print("\nDefining model signature and input example...")

# Define signature
sample_input = pd.DataFrame({
    'question': ['Who should I start this week: Josh Allen or Patrick Mahomes?']
})

sample_output = pd.DataFrame({
    'answer': ['Based on recent performance...'],
    'status': ['success']
})

signature = mlflow.models.infer_signature(sample_input, sample_output)

print("\n✓ Signature defined")
print("\nLogging model to MLflow with dictionary-based data...")

# Log the model with pre-loaded data as dictionaries (efficient serialization)
with mlflow.start_run(run_name="fantasai_chat_api_dict_data_v12") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="fantasai_chat_model",
        python_model=fantasai_model,
        signature=signature,
        input_example=sample_input,
        resources=[DatabricksServingEndpoint(endpoint_name=FOUNDATION_MODEL)],
        pip_requirements=[
            "databricks-sdk",
            "openai",
            "mlflow>=2.9.0",
            "lightgbm",
            "cloudpickle",
            "pandas",
            "numpy"
        ]
    )
    
    DEPLOYMENT_RUN_ID = run.info.run_id
    
    print("\n" + "="*70)
    print("✅ MODEL LOGGED WITH DICTIONARY-BASED DATA")
    print("="*70)
    print(f"Run ID: {DEPLOYMENT_RUN_ID}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"Data Access: Pre-loaded dictionaries (efficient serialization)")
    print(f"  - Recent stats: {len(recent_stats_dict['data'])} records")
    print(f"  - Player features: {len(player_features_dict['data'])} records")
    print(f"Models included: {list(loaded_models.keys())}")
    print(f"Packages: 7 dependencies")
    print("\nNext: Run Step 2 to update the serving endpoint")
    print("="*70)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput
import mlflow
import time

print("Step 2: Creating/Updating Model Serving Endpoint")
print("=" * 70)

# Initialize Workspace Client
w = WorkspaceClient()

ENDPOINT_NAME = "fantasai-chat-api"
model_uri = f"runs:/{DEPLOYMENT_RUN_ID}/fantasai_chat_model"

print(f"\nModel URI: {model_uri}")

# Register model to UC first
model_name = f"{CATALOG}.{SCHEMA}.fantasai_chat_model"
print(f"\nRegistering model to: {model_name}")

model_version = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"✓ Model registered: {model_name} version {model_version.version}")

# Check if endpoint exists
try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"\n✓ Endpoint '{ENDPOINT_NAME}' already exists")
    print(f"  Current status: {existing.state.config_update}")
    print(f"\n→ Updating endpoint to use new model version {model_version.version}...\n")
    
    # Update the endpoint with new model version
    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=model_name,
                entity_version=str(model_version.version),
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    )
    
    print(f"✓ Endpoint update initiated!")
    print(f"  Endpoint: {ENDPOINT_NAME}")
    print(f"  New model: {model_name} v{model_version.version}")
    print(f"\n⏳ Update will take 3-5 minutes.")
    print(f"\nNext: Wait for READY status, then run Step 3 to test")
    
except Exception as e:
    if "does not exist" in str(e) or "RESOURCE_DOES_NOT_EXIST" in str(e):
        # Endpoint doesn't exist, create it
        print(f"\nCreating new endpoint: {ENDPOINT_NAME}...\n")
        
        endpoint = w.serving_endpoints.create(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=model_name,
                        entity_version=str(model_version.version),
                        scale_to_zero_enabled=True,
                        workload_size="Small"
                    )
                ]
            )
        )
        
        print(f"✓ Endpoint creation initiated!")
        print(f"  Name: {endpoint.name}")
        print(f"  Model: {model_name} v{model_version.version}")
        print(f"\n⏳ This will take 5-10 minutes.")
        print(f"\nNext: Wait for READY status, then run Step 3")
    else:
        print(f"\n✗ Error: {e}")
        raise

In [0]:
import requests
import json
import time
from databricks.sdk.service.serving import EndpointStateReady

print("Step 3: Testing FantasAI Chat API Endpoint")
print("=" * 70)

# Get endpoint status
w = WorkspaceClient()
ENDPOINT_NAME = "fantasai-chat-api"

try:
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
    ready_status = endpoint.state.ready
    config_status = endpoint.state.config_update
    
    print(f"\nEndpoint Ready Status: {ready_status}")
    print(f"Config Status: {config_status}")
    
    # Check if ready (handle both enum and string comparisons)
    is_ready = (ready_status == EndpointStateReady.READY or 
                str(ready_status) == "EndpointStateReady.READY" or
                str(ready_status) == "READY")
    
    if not is_ready:
        print(f"\n⚠ Endpoint is not ready yet. Current status: {ready_status}")
        print(f"\nPlease wait for the endpoint to reach READY status before testing.")
        print(f"You can check status in the UI: Serving → {ENDPOINT_NAME}")
    else:
        print(f"\n✓ Endpoint is READY!\n")
        
        # Get workspace URL and token
        workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        
        endpoint_url = f"{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"
        
        print(f"Testing endpoint: {endpoint_url}\n")
        print("=" * 70)
        
        # Test 3: Wide Receiver Waiver Wire Pickup
        print("\n🏈 Test 3: Waiver Wire Pickup Decision\n")
        test_question_3 = "Should I pick up Jaxon Smith-Njigba from waivers? How has he been performing?"
        
        print(f"Q: {test_question_3}")
        print("\nCalling endpoint (this may take 30-60 seconds)...\n")
        
        response_3 = requests.post(
            endpoint_url,
            headers={
                "Authorization": f"Bearer {api_token}",
                "Content-Type": "application/json"
            },
            json={"inputs": {"question": [test_question_3]}},
            timeout=120
        )
        
        if response_3.status_code == 200:
            result_3 = response_3.json()
            print(f"A: {result_3.get('predictions', [{}])[0].get('answer', 'No answer')}")
            print(f"\n✓ Test 3 passed (status: {response_3.status_code})")
        else:
            print(f"\n✗ Test 3 failed (status: {response_3.status_code})")
            print(f"Error: {response_3.text}")
        
        print("\n" + "=" * 70)
        print("\n✅ DEPLOYMENT COMPLETE!\n")
        print(f"Your FantasAI Chat API is live at:")
        print(f"  {endpoint_url}")
        print(f"\nFront-End Integration:")
        print(f"  1. Use the endpoint URL above")
        print(f"  2. Generate a personal access token (User Settings → Developer → Access tokens)")
        print(f"  3. See Cell 11 for Python/JavaScript client examples")
        print(f"  4. Never expose your API token in front-end code - use a backend proxy")
        
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
print("Step 1: Logging FantasAI Chat Model to MLflow")
print("=" * 70)
print()

import mlflow.pyfunc

# Set MLflow tracking
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/kingoffrisco@yahoo.com/fantasai_chat_api_deployment")

with mlflow.start_run(run_name="fantasai_chat_api_v1") as run:
    print("Logging model to MLflow...")
    
    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FantasAIChatModel(),
        signature=signature,
        input_example=sample_input,
        pip_requirements=[
            "databricks-vectorsearch",
            "databricks-sdk",
            "openai",
            "mlflow",
            "lightgbm",
            "cloudpickle"
        ]
    )
    
    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    
    print(f"\n✅ Model logged successfully!")
    print("=" * 70)
    print(f"Run ID: {run_id}")
    print(f"Experiment ID: {experiment_id}")
    print(f"Model URI: runs:/{run_id}/model")
    print(f"\nArtifact Location: {run.info.artifact_uri}")
    print("=" * 70)
    print("\n✅ Ready for Step 2: Create Model Serving Endpoint")
    print(f"\nUse this Run ID for deployment: {run_id}")
    
# Store run_id for next step
DEPLOYMENT_RUN_ID = run_id
print(f"\nDeployment Run ID stored in variable: DEPLOYMENT_RUN_ID")

In [0]:
print("Step 2: Creating Model Serving Endpoint")
print("=" * 70)
print()

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

w = WorkspaceClient()

# Check if DEPLOYMENT_RUN_ID is set
try:
    run_id_to_deploy = DEPLOYMENT_RUN_ID
    print(f"Using Run ID: {run_id_to_deploy}")
except NameError:
    print("⚠️  Error: DEPLOYMENT_RUN_ID not found.")
    print("Please run the previous cell (Step 1) first to log the model.")
    raise

ENDPOINT_NAME = "fantasai-chat-api"

# Check if endpoint already exists
try:
    existing_endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"⚠️  Endpoint '{ENDPOINT_NAME}' already exists!")
    print(f"Current Status: {existing_endpoint.state.ready}")
    print("\nOptions:")
    print("1. Delete existing endpoint and recreate (use delete_endpoint() function)")
    print("2. Update existing endpoint with new model (use update_endpoint() function)")
    print(f"\nTo delete: w.serving_endpoints.delete(name='{ENDPOINT_NAME}')")
except Exception as e:
    # Endpoint doesn't exist, create it
    print(f"✅ Endpoint '{ENDPOINT_NAME}' does not exist. Creating new endpoint...")
    print()
    
    try:
        endpoint = w.serving_endpoints.create(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=f"runs:/{run_id_to_deploy}/model",
                        scale_to_zero_enabled=True,
                        workload_size="Small"
                    )
                ]
            )
        )
        
        print("=" * 70)
        print(f"✅ Endpoint '{endpoint.name}' created successfully!")
        print("=" * 70)
        print(f"\nEndpoint Name: {endpoint.name}")
        print(f"Status: Provisioning...")
        print(f"\n⏳ Please wait 5-10 minutes for endpoint to reach READY state.")
        print("\nYou can monitor status in Databricks UI:")
        print("  Serving → fantasai-chat-api")
        print("\nOr check programmatically:")
        print(f"  status = w.serving_endpoints.get(name='{ENDPOINT_NAME}')")
        print("  print(status.state.ready)")
        print("=" * 70)
        print("\n✅ Proceed to Step 3 after endpoint is READY")
        
    except Exception as create_error:
        print(f"❌ Error creating endpoint: {create_error}")
        raise

In [0]:
print("Step 3: Testing Model Serving Endpoint")
print("=" * 70)
print()

import requests
import time

# Get workspace info
api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

ENDPOINT_NAME = "fantasai-chat-api"

print(f"Workspace URL: {workspace_url}")
print(f"Endpoint Name: {ENDPOINT_NAME}")
print()

# Check endpoint status first
w = WorkspaceClient()
try:
    endpoint_status = w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Endpoint Status: {endpoint_status.state.ready}")
    print()
    
    if endpoint_status.state.ready != "READY":
        print("⚠️  Endpoint is not ready yet!")
        print(f"Current state: {endpoint_status.state.ready}")
        print("\nPlease wait for the endpoint to reach READY state before testing.")
        print("This typically takes 5-10 minutes.")
        print("\nYou can check status with:")
        print(f"  w.serving_endpoints.get(name='{ENDPOINT_NAME}').state.ready")
    else:
        print("✅ Endpoint is READY! Testing API...")
        print()
        
        # Test 1: Simple question
        print("Test 1: Simple QB comparison")
        print("-" * 70)
        
        url = f"{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"
        headers = {
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json"
        }
        
        test_question = "Who should I start this week: Josh Allen or Patrick Mahomes?"
        payload = {
            "inputs": {"question": [test_question]}
        }
        
        print(f"Question: {test_question}")
        print("\nSending request...")
        
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        
        if response.status_code == 200:
            result = response.json()
            print(f"\n✅ API Response (Status {response.status_code}):")
            print("=" * 70)
            
            # Parse response
            if 'predictions' in result:
                answer = result['predictions'][0].get('answer', 'No answer')
                status_msg = result['predictions'][0].get('status', 'unknown')
                print(f"Answer: {answer}")
                print(f"\nStatus: {status_msg}")
            else:
                print(result)
        else:
            print(f"\n❌ Error (Status {response.status_code}):")
            print(response.text)
        
        print("\n" + "=" * 70)
        
        # Test 2: Performance analysis
        print("\nTest 2: Player performance analysis")
        print("-" * 70)
        
        test_question2 = "Tell me about Travis Kelce's recent performance"
        payload2 = {
            "inputs": {"question": [test_question2]}
        }
        
        print(f"Question: {test_question2}")
        print("\nSending request...")
        
        response2 = requests.post(url, headers=headers, json=payload2, timeout=60)
        
        if response2.status_code == 200:
            result2 = response2.json()
            print(f"\n✅ API Response (Status {response2.status_code}):")
            print("=" * 70)
            
            if 'predictions' in result2:
                answer2 = result2['predictions'][0].get('answer', 'No answer')
                print(f"Answer: {answer2[:300]}...")  # Truncate for display
            else:
                print(result2)
        else:
            print(f"\n❌ Error (Status {response2.status_code}):")
            print(response2.text)
        
        print("\n" + "=" * 70)
        print("✅ Testing complete! Endpoint is working.")
        print("=" * 70)
        print("\nEndpoint URL for front-end:")
        print(f"{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations")
        print("\nSee Cell 11 for Python/JavaScript integration examples.")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure the endpoint exists. Run Step 2 if not created yet.")

In [0]:
# how to print the output
{
  "predictions": [
    {
      "answer": "Rogers is a player with a status of 'active'.",
      "status": "active"
    }
  ]
}

In [0]:
# ====================================================================
# CLOUDFLARE PIPELINE: nflverse Data Fetcher for R2 Storage
# ====================================================================
# This script fetches NFL data from nflverse and formats it for your
# FantasAI model. Export to JSON and upload to CloudFlare R2.

import requests
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

print("FantasAI CloudFlare Data Pipeline")
print("=" * 70)
print("Data Source: nflverse (free, no auth required)")
print("Output: JSON files ready for R2 storage\n")

# ====================================================================
# STEP 1: Fetch Player Stats from nflverse
# ====================================================================

def fetch_nflverse_weekly_stats(season=2025, weeks=None):
    """
    Fetch weekly player stats from nflverse.
    
    Args:
        season: NFL season year (default 2025)
        weeks: List of weeks to fetch (default: all available)
    
    Returns:
        DataFrame with player stats
    """
    print(f"Fetching {season} season data from nflverse...")
    
    # nflverse weekly player stats endpoint
    # Note: 2025 data may not be available yet - falls back to 2024
    url = f"https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_{season}.csv"
    
    try:
        df = pd.read_csv(url)
        print(f"  ✓ Loaded {len(df):,} records")
        
        # Filter by weeks if specified
        if weeks:
            df = df[df['week'].isin(weeks)]
            print(f"  ✓ Filtered to weeks {weeks}: {len(df):,} records")
        
        return df
        
    except Exception as e:
        print(f"  ✗ Error fetching {season} data: {e}")
        print(f"  → Trying {season-1} as fallback...")
        
        # Fallback to previous season
        url_fallback = f"https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_{season-1}.csv"
        df = pd.read_csv(url_fallback)
        print(f"  ✓ Loaded {len(df):,} records from {season-1}")
        return df

# Fetch recent weeks
raw_stats = fetch_nflverse_weekly_stats(season=2025, weeks=[15, 16, 17, 18])

print(f"\nAvailable columns: {list(raw_stats.columns[:10])}...")
print(f"Positions: {raw_stats['position'].unique()[:20]}")

# ====================================================================
# STEP 2: Transform to FantasAI Format
# ====================================================================

def transform_to_fantasai_recent_stats(df):
    """
    Transform nflverse data to FantasAI recent_stats format.
    
    Output schema:
        - player_name
        - position
        - team
        - points (fantasy_points)
        - season
        - week
    """
    print("\nTransforming to recent_stats format...")
    
    # Map nflverse columns to FantasAI schema
    # nflverse uses: player_display_name, position, recent_team, fantasy_points_ppr
    transformed = pd.DataFrame({
        'player_name': df.get('player_display_name', df.get('player_name', '')),
        'position': df['position'],
        'team': df.get('recent_team', df.get('team', '')),
        'points': df.get('fantasy_points_ppr', df.get('fantasy_points', 0)),
        'season': df['season'],
        'week': df['week']
    })
    
    # Filter to QB, RB, WR, TE only
    transformed = transformed[transformed['position'].isin(['QB', 'RB', 'WR', 'TE'])]
    
    # Sort by recent performance
    transformed = transformed.sort_values(['week', 'points'], ascending=[False, False])
    
    # Limit to top 100 recent performers
    transformed = transformed.head(100)
    
    print(f"  ✓ Transformed {len(transformed)} records")
    print(f"  ✓ Positions: {transformed['position'].value_counts().to_dict()}")
    
    return transformed

recent_stats_df = transform_to_fantasai_recent_stats(raw_stats)

# ====================================================================
# STEP 3: Engineer ML Features
# ====================================================================

def engineer_ml_features(df):
    """
    Engineer 42 features for ML prediction.
    
    Features include:
    - Rolling averages (3g, 5g, 10g)
    - Momentum & trends
    - Position-specific stats
    - Opponent history
    - Team strength metrics
    """
    print("\nEngineering ML features...")
    
    # Sort by player and week
    df = df.sort_values(['player_name', 'season', 'week'])
    
    # Group by player for rolling calculations
    features_list = []
    
    for player_name, player_df in df.groupby('player_name'):
        player_df = player_df.copy()
        
        # Rolling statistics
        player_df['rolling_3g_avg'] = player_df['points'].rolling(3, min_periods=1).mean()
        player_df['rolling_5g_avg'] = player_df['points'].rolling(5, min_periods=1).mean()
        player_df['rolling_10g_avg'] = player_df['points'].rolling(10, min_periods=1).mean()
        player_df['rolling_5g_stddev'] = player_df['points'].rolling(5, min_periods=1).std().fillna(0)
        
        # Season average to date
        player_df['season_avg_to_date'] = player_df.groupby('season')['points'].expanding().mean().reset_index(drop=True)
        
        # Momentum metrics
        player_df['momentum_score'] = player_df['rolling_3g_avg'] - player_df['rolling_5g_avg']
        player_df['wow_change'] = player_df['points'].diff().fillna(0)
        
        # Scoring streak (consecutive games > 10 pts)
        player_df['scoring_streak'] = (player_df['points'] > 10).astype(int).groupby(
            (player_df['points'] <= 10).cumsum()
        ).cumsum()
        
        # Form variance ratio
        player_df['form_variance_ratio'] = player_df['rolling_5g_stddev'] / (player_df['rolling_5g_avg'] + 0.1)
        
        features_list.append(player_df)
    
    df_with_features = pd.concat(features_list, ignore_index=True)
    
    # Add time-based features
    df_with_features['is_early_season'] = (df_with_features['week'] <= 6).astype(int)
    df_with_features['is_mid_season'] = ((df_with_features['week'] > 6) & (df_with_features['week'] <= 12)).astype(int)
    df_with_features['is_late_season'] = ((df_with_features['week'] > 12) & (df_with_features['week'] <= 17)).astype(int)
    df_with_features['is_playoff_weeks'] = (df_with_features['week'] >= 15).astype(int)
    df_with_features['weeks_into_season'] = df_with_features['week']
    
    # Games played
    df_with_features['games_played_streak'] = df_with_features.groupby('player_name').cumcount() + 1
    df_with_features['weeks_since_last_game'] = 0  # Assume no gaps (can enhance)
    df_with_features['coming_off_bye'] = 0  # Would need schedule data
    
    # Position-specific stats (need to map from raw data)
    # For now, set defaults - you'd join with detailed stats
    position_defaults = {
        'qb_passing_yards': 0, 'qb_passing_tds': 0, 'qb_attempts': 0, 'qb_completions': 0,
        'rb_carries': 0, 'rb_rushing_yards': 0, 'rb_rushing_tds': 0,
        'rec_targets': 0, 'rec_receptions': 0, 'rec_yards': 0, 'rec_tds': 0,
        'rolling_3g_targets': 0, 'rolling_3g_carries': 0, 'rolling_3g_attempts': 0
    }
    
    for col, default_val in position_defaults.items():
        if col not in df_with_features.columns:
            df_with_features[col] = default_val
    
    # Opponent history metrics
    df_with_features['career_avg_vs_opponent'] = df_with_features['season_avg_to_date']
    df_with_features['games_vs_opponent'] = 1
    df_with_features['max_points_vs_opponent'] = df_with_features['points']
    df_with_features['recent_avg_vs_opponent'] = df_with_features['rolling_3g_avg']
    
    # Team strength metrics (would need team-level data)
    df_with_features['team_offensive_strength'] = 100  # Default mid-range
    df_with_features['team_offense_rank'] = 16
    df_with_features['position_share_pct'] = 20.0
    
    # Defensive matchup (would need opponent defense data)
    df_with_features['def_points_allowed_avg'] = 15.0
    df_with_features['def_rank_vs_position'] = 16
    
    # Season rankings
    df_with_features['season_position_rank'] = df_with_features.groupby(['season', 'position'])['season_avg_to_date'].rank(ascending=False, method='dense')
    df_with_features['season_percentile'] = df_with_features.groupby(['season', 'position'])['season_avg_to_date'].rank(pct=True)
    
    print(f"  ✓ Engineered features for {len(df_with_features)} records")
    print(f"  ✓ Total features: {len(df_with_features.columns)}")
    
    return df_with_features

player_features_df = engineer_ml_features(raw_stats)

# ====================================================================
# STEP 4: Export to JSON for R2 Storage
# ====================================================================

def export_to_json(recent_stats_df, player_features_df, output_dir="/tmp"):
    """
    Export DataFrames to JSON format for CloudFlare R2.
    """
    print("\nExporting to JSON...")
    
    # Convert to dictionary format (matches current model)
    recent_stats_dict = {
        'data': recent_stats_df.to_dict('records'),
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'record_count': len(recent_stats_df),
            'source': 'nflverse'
        }
    }
    
    player_features_dict = {
        'data': player_features_df.head(500).to_dict('records'),  # Limit to 500
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'record_count': len(player_features_df.head(500)),
            'source': 'nflverse',
            'feature_count': 42
        }
    }
    
    # Save to files
    recent_stats_path = f"{output_dir}/recent_stats.json"
    player_features_path = f"{output_dir}/player_features.json"
    
    with open(recent_stats_path, 'w') as f:
        json.dump(recent_stats_dict, f, indent=2)
    print(f"  ✓ Saved recent_stats: {recent_stats_path}")
    print(f"    Size: {len(json.dumps(recent_stats_dict)) / 1024:.1f} KB")
    
    with open(player_features_path, 'w') as f:
        json.dump(player_features_dict, f, indent=2)
    print(f"  ✓ Saved player_features: {player_features_path}")
    print(f"    Size: {len(json.dumps(player_features_dict)) / 1024:.1f} KB")
    
    return recent_stats_path, player_features_path

recent_path, features_path = export_to_json(recent_stats_df, player_features_df)

print("\n" + "=" * 70)
print("✅ PIPELINE COMPLETE!")
print("=" * 70)
print("\nNext Steps for CloudFlare:")
print("1. Upload JSON files to R2 bucket")
print("2. Export LightGBM models (see next cell)")
print("3. Adapt FantasAIChatModel for CloudFlare Workers")
print("4. Replace OpenAI endpoint with your LLM provider")
print("\nSchedule: Run this weekly after games complete (Tuesday)")

In [0]:
# ====================================================================
# CLOUDFLARE WORKER: Deployment Configuration
# ====================================================================

import os

R2_ENDPOINT = "https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com"
R2_BUCKET_NAME = "fantasai-r2"

print("CloudFlare Worker Deployment Configuration")
print("=" * 70)
print(f"\nR2 Endpoint: {R2_ENDPOINT}")
print(f"Bucket Name: {R2_BUCKET_NAME}\n")

# ====================================================================
# Create wrangler.toml Configuration
# ====================================================================

wrangler_config = '''# ====================================================================
# Wrangler Configuration for FantasAI Chat API
# ====================================================================

name = "fantasai-chat-api"
main = "src/index.js"
compatibility_date = "2024-01-01"

# R2 Bucket Binding
[[r2_buckets]]
binding = "R2_BUCKET"
bucket_name = "fantasai-r2"

# Environment Variables
[vars]
R2_ENDPOINT = "https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com"

# Secrets (set via CLI: wrangler secret put OPENAI_API_KEY)
# OPENAI_API_KEY = "your-key-here" (use wrangler secret put instead)
'''

config_path = "/tmp/cloudflare_models/wrangler.toml"
with open(config_path, 'w') as f:
    f.write(wrangler_config)

print(f"✓ Created: {config_path}")
print("\n" + wrangler_config)

# ====================================================================
# Create Enhanced Worker Script
# ====================================================================

enhanced_worker = '''// ====================================================================
// CLOUDFLARE WORKER: FantasAI Chat API
// R2 Endpoint: https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com
// Bucket: fantasai-r2
// ====================================================================

export default {
  async fetch(request, env) {
    // CORS headers for cross-origin requests
    const corsHeaders = {
      'Access-Control-Allow-Origin': '*',
      'Access-Control-Allow-Methods': 'POST, OPTIONS',
      'Access-Control-Allow-Headers': 'Content-Type, Authorization',
    };

    // Handle CORS preflight
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: corsHeaders });
    }

    // Only accept POST requests
    if (request.method !== 'POST') {
      return new Response('Method not allowed', { 
        status: 405, 
        headers: corsHeaders 
      });
    }

    try {
      // Parse incoming request
      const body = await request.json();
      const question = body.inputs?.question?.[0] || body.question;

      if (!question) {
        return new Response(
          JSON.stringify({ 
            predictions: [{ 
              answer: 'Missing question parameter', 
              status: 'error' 
            }] 
          }), 
          { status: 400, headers: { ...corsHeaders, 'Content-Type': 'application/json' } }
        );
      }

      console.log(`Processing question: ${question}`);

      // ================================================================
      // STEP 1: Load Data from R2
      // ================================================================
      
      const [recentStatsObj, playerFeaturesObj] = await Promise.all([
        env.R2_BUCKET.get('recent_stats.json'),
        env.R2_BUCKET.get('player_features.json')
      ]);

      if (!recentStatsObj || !playerFeaturesObj) {
        throw new Error('Failed to load data from R2');
      }

      const recentStatsData = JSON.parse(await recentStatsObj.text());
      const playerFeaturesData = JSON.parse(await playerFeaturesObj.text());

      console.log(`Loaded ${recentStatsData.data.length} recent stats, ${playerFeaturesData.data.length} player features`);

      // ================================================================
      // STEP 2: Search for Relevant Players
      // ================================================================
      
      // Simple keyword search for player names
      const questionLower = question.toLowerCase();
      const relevantPlayers = recentStatsData.data.filter(player => 
        questionLower.includes(player.player_name.toLowerCase())
      ).slice(0, 5);

      // ================================================================
      // STEP 3: Generate Context for LLM
      // ================================================================
      
      let contextMessage = `You are FantasAI, a fantasy football expert assistant.\\n\\n`;
      
      if (relevantPlayers.length > 0) {
        contextMessage += `Recent player performance:\\n`;
        relevantPlayers.forEach(p => {
          contextMessage += `- ${p.player_name} (${p.position}, ${p.team}): ${p.points} pts (Week ${p.week})\\n`;
        });
      } else {
        // Show top performers from recent data
        contextMessage += `Top recent performers:\\n`;
        recentStatsData.data.slice(0, 5).forEach(p => {
          contextMessage += `- ${p.player_name} (${p.position}, ${p.team}): ${p.points} pts (Week ${p.week})\\n`;
        });
      }

      // ================================================================
      // STEP 4: Call LLM (OpenAI, Anthropic, or other)
      // ================================================================
      
      // NOTE: LightGBM models require external inference API or WASM
      // For now, using LLM with contextual data
      // You can add model predictions by calling an external inference API
      
      const llmResponse = await fetch('https://api.openai.com/v1/chat/completions', {
        method: 'POST',
        headers: {
          'Authorization': `Bearer ${env.OPENAI_API_KEY}`,
          'Content-Type': 'application/json'
        },
        body: JSON.stringify({
          model: 'gpt-4',
          messages: [
            { role: 'system', content: contextMessage },
            { role: 'user', content: question }
          ],
          temperature: 0.7,
          max_tokens: 500
        })
      });

      if (!llmResponse.ok) {
        throw new Error(`LLM API error: ${llmResponse.statusText}`);
      }

      const llmData = await llmResponse.json();
      const answer = llmData.choices[0].message.content;

      // ================================================================
      // STEP 5: Return Response (Databricks-compatible format)
      // ================================================================
      
      return new Response(
        JSON.stringify({
          predictions: [{
            answer: answer,
            status: 'success',
            metadata: {
              players_found: relevantPlayers.length,
              timestamp: new Date().toISOString()
            }
          }]
        }),
        { 
          status: 200, 
          headers: { ...corsHeaders, 'Content-Type': 'application/json' }
        }
      );

    } catch (error) {
      console.error('Error:', error);
      
      return new Response(
        JSON.stringify({ 
          predictions: [{
            answer: `Error processing request: ${error.message}`,
            status: 'error'
          }]
        }),
        { 
          status: 500, 
          headers: { 
            ...corsHeaders, 
            'Content-Type': 'application/json' 
          }
        }
      );
    }
  }
};
'''

worker_path = "/tmp/cloudflare_models/src/index.js"
os.makedirs("/tmp/cloudflare_models/src", exist_ok=True)
with open(worker_path, 'w') as f:
    f.write(enhanced_worker)

print(f"✓ Created: {worker_path}")

# ====================================================================
# Deployment Commands
# ====================================================================

print("\n" + "=" * 70)
print("DEPLOYMENT COMMANDS")
print("=" * 70)
print("""
# 1. Navigate to project directory
cd /tmp/cloudflare_models

# 2. Install Wrangler (if not already installed)
npm install -g wrangler

# 3. Login to CloudFlare
wrangler login

# 4. Create R2 bucket (if not exists)
wrangler r2 bucket create fantasai-r2

# 5. Upload data files to R2 (see Cell 22 for details)

# 6. Set secrets
wrangler secret put OPENAI_API_KEY
# (paste your OpenAI API key when prompted)

# 7. Deploy worker
wrangler deploy

# 8. Test endpoint
curl -X POST https://fantasai-chat-api.YOUR_SUBDOMAIN.workers.dev \\
  -H "Content-Type: application/json" \\
  -d '{"inputs": {"question": ["Who should I start: Josh Allen or Patrick Mahomes?"]}}'\n""")

print("=" * 70)
print("✅ CONFIGURATION COMPLETE")
print("=" * 70)
print("\nGenerated files:")
print("  • wrangler.toml (CloudFlare config)")
print("  • src/index.js (Worker script)")
print("\nNext: Upload to R2 (Cell 22) then deploy with 'wrangler deploy'")

# ====================================================================
# Alternative: Python-based R2 Upload Script
# ====================================================================

print("\n\n" + "=" * 70)
print("BONUS: Python R2 Upload Script")
print("=" * 70)

upload_script = '''#!/usr/bin/env python3
# ====================================================================
# Python Script: Upload Files to CloudFlare R2
# ====================================================================

import boto3
import os
from pathlib import Path

# R2 Configuration
R2_ENDPOINT = "https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com"
R2_BUCKET = "fantasai-r2"

# Set these environment variables or replace with your keys
R2_ACCESS_KEY = os.getenv("R2_ACCESS_KEY_ID")
R2_SECRET_KEY = os.getenv("R2_SECRET_ACCESS_KEY")

if not R2_ACCESS_KEY or not R2_SECRET_KEY:
    print("Error: Set R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY environment variables")
    exit(1)

# Initialize S3 client (R2 is S3-compatible)
s3 = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY
)

# Files to upload
files = [
    ("/tmp/recent_stats.json", "recent_stats.json"),
    ("/tmp/player_features.json", "player_features.json"),
    ("/tmp/cloudflare_models/qb_model.pkl", "models/qb_model.pkl"),
    ("/tmp/cloudflare_models/rb_model.pkl", "models/rb_model.pkl"),
    ("/tmp/cloudflare_models/wr_model.pkl", "models/wr_model.pkl"),
    ("/tmp/cloudflare_models/te_model.pkl", "models/te_model.pkl"),
    ("/tmp/cloudflare_models/model_metadata.json", "model_metadata.json"),
]

print(f"Uploading to R2 bucket: {R2_BUCKET}")
print(f"Endpoint: {R2_ENDPOINT}\\n")

for local_path, r2_key in files:
    if not os.path.exists(local_path):
        print(f"⚠️  Skipping {local_path} (not found)")
        continue
    
    try:
        file_size = os.path.getsize(local_path) / 1024
        print(f"Uploading {r2_key} ({file_size:.1f} KB)...")
        
        s3.upload_file(local_path, R2_BUCKET, r2_key)
        print(f"  ✓ Uploaded successfully\\n")
        
    except Exception as e:
        print(f"  ✗ Error: {e}\\n")

print("✅ Upload complete!")
'''

upload_script_path = "/tmp/cloudflare_models/upload_to_r2.py"
with open(upload_script_path, 'w') as f:
    f.write(upload_script)

os.chmod(upload_script_path, 0o755)  # Make executable

print(f"✓ Created: {upload_script_path}")
print("\nUsage:")
print("  export R2_ACCESS_KEY_ID='your-access-key'")
print("  export R2_SECRET_ACCESS_KEY='your-secret-key'")
print("  python /tmp/cloudflare_models/upload_to_r2.py")
print("\n(Get R2 credentials from CloudFlare Dashboard → R2 → Manage R2 API Tokens)")

In [0]:
# ====================================================================
# CLOUDFLARE PIPELINE: Export LightGBM Models for R2 Storage
# ====================================================================

import mlflow
import pickle
import json
import os

print("Exporting LightGBM Models for CloudFlare R2")
print("=" * 70)

# Model Run IDs from Phase 3
MODEL_RUN_IDS = {
    'QB': '912be789d4524a0a9f926431fc8a88fd',
    'RB': 'ae3a7c9fb600494aa93efd7cc22ef7cb',
    'WR': '237753d24cc24d29a85a7cdf526a5780',
    'TE': 'c18fcbf8c7b14aa7a8a1a47760048fe0'
}

FEATURE_COLS = [
    'rolling_3g_avg', 'rolling_5g_avg', 'season_avg_to_date', 'momentum_score',
    'wow_change', 'scoring_streak', 'rolling_10g_avg', 'rolling_5g_stddev',
    'form_variance_ratio', 'is_early_season', 'is_mid_season', 'is_late_season',
    'is_playoff_weeks', 'weeks_into_season', 'games_played_streak',
    'weeks_since_last_game', 'coming_off_bye', 'qb_passing_yards', 'qb_passing_tds',
    'qb_attempts', 'qb_completions', 'rb_carries', 'rb_rushing_yards',
    'rb_rushing_tds', 'rec_targets', 'rec_receptions', 'rec_yards', 'rec_tds',
    'rolling_3g_targets', 'rolling_3g_carries', 'rolling_3g_attempts',
    'career_avg_vs_opponent', 'games_vs_opponent', 'max_points_vs_opponent',
    'recent_avg_vs_opponent', 'team_offensive_strength', 'team_offense_rank',
    'position_share_pct', 'def_points_allowed_avg', 'def_rank_vs_position',
    'season_position_rank', 'season_percentile'
]

output_dir = "/tmp/cloudflare_models"
os.makedirs(output_dir, exist_ok=True)

# ====================================================================
# Export LightGBM Models
# ====================================================================

print("\nExporting LightGBM models...\n")

exported_models = {}

for position, run_id in MODEL_RUN_IDS.items():
    print(f"Exporting {position} model...")
    
    try:
        # Load model from MLflow
        model_uri = f"runs:/{run_id}/model"
        model = mlflow.lightgbm.load_model(model_uri)
        
        # Save as pickle file
        model_path = f"{output_dir}/{position.lower()}_model.pkl"
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        
        file_size = os.path.getsize(model_path) / 1024
        print(f"  ✓ Saved: {model_path}")
        print(f"    Size: {file_size:.1f} KB")
        print(f"    Run ID: {run_id}\n")
        
        exported_models[position] = {
            'path': model_path,
            'run_id': run_id,
            'size_kb': file_size
        }
        
    except Exception as e:
        print(f"  ✗ Error exporting {position} model: {e}\n")

# ====================================================================
# Export Model Metadata
# ====================================================================

print("Exporting model metadata...\n")

metadata = {
    'models': {
        position: {
            'run_id': info['run_id'],
            'filename': f"{position.lower()}_model.pkl",
            'size_kb': info['size_kb']
        }
        for position, info in exported_models.items()
    },
    'feature_columns': FEATURE_COLS,
    'feature_count': len(FEATURE_COLS),
    'positions': list(MODEL_RUN_IDS.keys()),
    'exported_at': datetime.now().isoformat(),
    'model_type': 'lightgbm',
    'framework_version': 'lightgbm>=3.3.0'
}

metadata_path = f"{output_dir}/model_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved metadata: {metadata_path}")
print(f"  Feature columns: {len(FEATURE_COLS)}")
print(f"  Models: {', '.join(exported_models.keys())}")

# ====================================================================
# Create CloudFlare Worker Template
# ====================================================================

print("\nCreating CloudFlare Worker template...\n")

worker_template = '''// ====================================================================
// CLOUDFLARE WORKER: FantasAI Chat API
// ====================================================================

export default {
  async fetch(request, env) {
    // CORS headers
    const corsHeaders = {
      'Access-Control-Allow-Origin': '*',
      'Access-Control-Allow-Methods': 'POST, OPTIONS',
      'Access-Control-Allow-Headers': 'Content-Type, Authorization',
    };

    // Handle CORS preflight
    if (request.method === 'OPTIONS') {
      return new Response(null, { headers: corsHeaders });
    }

    if (request.method !== 'POST') {
      return new Response('Method not allowed', { status: 405 });
    }

    try {
      // Parse request
      const body = await request.json();
      const question = body.inputs?.question?.[0] || body.question;

      if (!question) {
        return new Response(
          JSON.stringify({ error: 'Missing question' }), 
          { status: 400, headers: { ...corsHeaders, 'Content-Type': 'application/json' } }
        );
      }

      // Load data from R2
      const recentStats = await env.R2_BUCKET.get('recent_stats.json');
      const playerFeatures = await env.R2_BUCKET.get('player_features.json');
      const models = {
        QB: await env.R2_BUCKET.get('qb_model.pkl'),
        RB: await env.R2_BUCKET.get('rb_model.pkl'),
        WR: await env.R2_BUCKET.get('wr_model.pkl'),
        TE: await env.R2_BUCKET.get('te_model.pkl')
      };

      // Parse JSON data
      const recentStatsData = JSON.parse(await recentStats.text());
      const playerFeaturesData = JSON.parse(await playerFeatures.text());

      // TODO: Implement prediction logic
      // 1. Search for players in question
      // 2. Extract features
      // 3. Load and run LightGBM models (requires WASM or external API)
      // 4. Generate LLM response with predictions

      // For now, call external LLM API (OpenAI, Anthropic, etc.)
      const llmResponse = await fetch('https://api.openai.com/v1/chat/completions', {
        method: 'POST',
        headers: {
          'Authorization': `Bearer ${env.OPENAI_API_KEY}`,
          'Content-Type': 'application/json'
        },
        body: JSON.stringify({
          model: 'gpt-4',
          messages: [
            {
              role: 'system',
              content: `You are FantasAI, a fantasy football expert. Use the provided data to answer questions.\n\nRecent Stats: ${JSON.stringify(recentStatsData.data.slice(0, 5))}`
            },
            { role: 'user', content: question }
          ]
        })
      });

      const llmData = await llmResponse.json();
      const answer = llmData.choices[0].message.content;

      // Return response in Databricks format for compatibility
      return new Response(
        JSON.stringify({
          predictions: [{
            answer: answer,
            status: 'success'
          }]
        }),
        { 
          status: 200, 
          headers: { ...corsHeaders, 'Content-Type': 'application/json' }
        }
      );

    } catch (error) {
      return new Response(
        JSON.stringify({ 
          predictions: [{
            answer: 'Error processing request',
            status: 'error',
            error: error.message
          }]
        }),
        { status: 500, headers: { ...corsHeaders, 'Content-Type': 'application/json' } }
      );
    }
  }
};
'''

worker_path = f"{output_dir}/worker.js"
with open(worker_path, 'w') as f:
    f.write(worker_template)

print(f"✓ Saved CloudFlare Worker template: {worker_path}")

# ====================================================================
# Summary
# ====================================================================

print("\n" + "=" * 70)
print("✅ MODEL EXPORT COMPLETE!")
print("=" * 70)
print(f"\nExported files in: {output_dir}/")
print("\nFiles:")
for position in exported_models.keys():
    print(f"  • {position.lower()}_model.pkl")
print(f"  • model_metadata.json")
print(f"  • worker.js (template)")

print("\nNext Steps:")
print("1. Upload all .pkl files to R2 bucket")
print("2. Upload recent_stats.json and player_features.json to R2")
print("3. Deploy worker.js to CloudFlare Workers")
print("4. Set environment variables:")
print("   - R2_BUCKET (binding to your R2 bucket)")
print("   - OPENAI_API_KEY (or your LLM provider key)")
print("\nNote: LightGBM models in CloudFlare Workers require:")
print("  Option A: WASM build of LightGBM (experimental)")
print("  Option B: Call external inference API")
print("  Option C: Convert models to ONNX format")
print("\nRecommendation: Use external inference API initially")

In [0]:
# ====================================================================
# CLOUDFLARE R2: Upload Instructions & Configuration
# ====================================================================

import os

R2_ENDPOINT = "https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com"
R2_BUCKET_NAME = "fantasai-r2"

print("CloudFlare R2 Upload Guide")
print("=" * 70)
print(f"\nR2 Endpoint: {R2_ENDPOINT}")
print(f"Bucket Name: {R2_BUCKET_NAME}")
print("\n" + "=" * 70)

# ====================================================================
# Files to Upload
# ====================================================================

print("\nFILES TO UPLOAD TO R2:")
print("=" * 70)

files_to_upload = [
    {
        'file': '/tmp/recent_stats.json',
        'r2_key': 'recent_stats.json',
        'description': 'Recent player stats (100 records)'
    },
    {
        'file': '/tmp/player_features.json',
        'r2_key': 'player_features.json',
        'description': 'ML features (500 records)'
    },
    {
        'file': '/tmp/cloudflare_models/qb_model.pkl',
        'r2_key': 'models/qb_model.pkl',
        'description': 'QB LightGBM model'
    },
    {
        'file': '/tmp/cloudflare_models/rb_model.pkl',
        'r2_key': 'models/rb_model.pkl',
        'description': 'RB LightGBM model'
    },
    {
        'file': '/tmp/cloudflare_models/wr_model.pkl',
        'r2_key': 'models/wr_model.pkl',
        'description': 'WR LightGBM model'
    },
    {
        'file': '/tmp/cloudflare_models/te_model.pkl',
        'r2_key': 'models/te_model.pkl',
        'description': 'TE LightGBM model'
    },
    {
        'file': '/tmp/cloudflare_models/model_metadata.json',
        'r2_key': 'model_metadata.json',
        'description': 'Model metadata & feature columns'
    }
]

for item in files_to_upload:
    print(f"\n{item['description']}:")
    print(f"  Local:  {item['file']}")
    print(f"  R2 Key: {item['r2_key']}")

# ====================================================================
# Upload Methods
# ====================================================================

print("\n\n" + "=" * 70)
print("UPLOAD METHODS:")
print("=" * 70)

print("""

METHOD 1: CloudFlare Dashboard (Easiest)
-----------------------------------------
1. Go to CloudFlare Dashboard → R2
2. Create or select bucket: fantasai-r2
3. Click "Upload files"
4. Upload all 7 files listed above


METHOD 2: Wrangler CLI (Recommended for automation)
----------------------------------------------------
# Install Wrangler
npm install -g wrangler

# Login to CloudFlare
wrangler login

# Create bucket (if not exists)
wrangler r2 bucket create fantasai-r2

# Upload files
wrangler r2 object put fantasai-r2/recent_stats.json --file=/tmp/recent_stats.json
wrangler r2 object put fantasai-r2/player_features.json --file=/tmp/player_features.json
wrangler r2 object put fantasai-r2/models/qb_model.pkl --file=/tmp/cloudflare_models/qb_model.pkl
wrangler r2 object put fantasai-r2/models/rb_model.pkl --file=/tmp/cloudflare_models/rb_model.pkl
wrangler r2 object put fantasai-r2/models/wr_model.pkl --file=/tmp/cloudflare_models/wr_model.pkl
wrangler r2 object put fantasai-r2/models/te_model.pkl --file=/tmp/cloudflare_models/te_model.pkl
wrangler r2 object put fantasai-r2/model_metadata.json --file=/tmp/cloudflare_models/model_metadata.json


METHOD 3: AWS CLI (S3-compatible)
----------------------------------
# Configure AWS CLI with R2 credentials
# Get Access Key ID and Secret from CloudFlare Dashboard → R2 → Manage R2 API Tokens

aws s3 cp /tmp/recent_stats.json s3://fantasai-r2/recent_stats.json \\
  --endpoint-url https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com

aws s3 cp /tmp/player_features.json s3://fantasai-r2/player_features.json \\
  --endpoint-url https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com

aws s3 cp /tmp/cloudflare_models/ s3://fantasai-r2/models/ \\
  --recursive --endpoint-url https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com


METHOD 4: Python boto3 (Programmatic)
--------------------------------------
""")

print("""
import boto3

s3 = boto3.client(
    's3',
    endpoint_url='https://a121da9c5331b463f2c3651bd2e6e789.r2.cloudflarestorage.com',
    aws_access_key_id='YOUR_R2_ACCESS_KEY_ID',
    aws_secret_access_key='YOUR_R2_SECRET_ACCESS_KEY'
)

# Upload files
s3.upload_file('/tmp/recent_stats.json', 'fantasai-r2', 'recent_stats.json')
s3.upload_file('/tmp/player_features.json', 'fantasai-r2', 'player_features.json')
s3.upload_file('/tmp/cloudflare_models/qb_model.pkl', 'fantasai-r2', 'models/qb_model.pkl')
# ... etc
""")

print("\n" + "=" * 70)
print("NEXT STEPS AFTER UPLOAD:")
print("=" * 70)
print("""
1. Run Cell 19 to generate JSON data files
2. Run Cell 21 to export LightGBM models
3. Upload all files to R2 using one of the methods above
4. Deploy CloudFlare Worker (see Cell 22)
5. Test the API endpoint
""")

print("\n💡 TIP: Set up weekly automation to regenerate data files from nflverse")
print("    Schedule Cell 19 to run every Tuesday after games complete")